# Programming 101
**RGS Intro to Machine Learning** · ~75 minute lecture

By the end of this lecture you should be able to:

1. Create variables and recognize the basic types of objects (strings, ints, floats, lists, arrays, DataFrames)
2. Explain what a *reference* (pointer) is and why it sometimes surprises you
3. Write and call your own functions, and call methods on objects
4. Load data from a file using a path
5. Explain why `0.1 + 0.2 != 0.3` and how to compare floats safely


*Run each cell with* `Shift+Enter`. *Change things and see what breaks; that is the best way to learn.*

In [ ]:
# We will use three libraries throughout
import numpy as np
import pandas as pd
from pathlib import Path

---
# 1. Variables and objects

A **variable** is a *name* that refers to a value. A value in memory (a number, a piece of text, a table) is an **object**.
We create variables with the **assignment operator** `=`:  `name = value`.

In [ ]:
x = 5
print(x)

Every object has a **type**, which determines what you can do with it. Use `type()` to ask.

In [ ]:
type(x)

## 1.1 Basic types

| Type | Python name | Example | Used for |
|---|---|---|---|
| Integer | `int` | `42` | counts, indices |
| Floating point | `float` | `3.14` | real-valued measurements |
| String | `str` | `"hello"` | text, labels, file names |
| Boolean | `bool` | `True` | yes/no, conditions |

In [ ]:
n_students = 24          # int
height_m   = 1.78        # float
name       = "Ada"       # str  (single or double quotes both work)
is_enrolled = True       # bool

for obj in [n_students, height_m, name, is_enrolled]:
    print(repr(obj), "->", type(obj).__name__)

Operators behave according to the type. `+` adds numbers but *concatenates* strings.

In [ ]:
print(2 + 3)
print("2" + "3")
print(2 * 3, "|", "ab" * 3)
print(7 / 2, 7 // 2, 7 % 2)    # true division, floor division, remainder

**Naming rules:** letters, digits, underscores; can't start with a digit; case-sensitive. Use descriptive `snake_case` names. Reassigning a name is fine; the name just points at a new object.

In [ ]:
x = 5
x = x + 1      # read: compute x + 1, then point the name x at the result
x

**Converting between types** is called *casting*.

In [ ]:
print(int("12") + 1)
print(float("3.5") * 2)
print(str(99) + " problems")
print(int(3.9))          # truncates toward zero, does NOT round

## 1.2 Collections: lists and vectors

A **list** is an ordered, mutable collection of anything. Indexing starts at **0**.

In [ ]:
scores = [88, 92, 79, 95]
print(scores[0], scores[-1])     # first and last
print(scores[1:3])               # slice: start inclusive, stop exclusive
scores.append(60)
print(scores, len(scores))

Lists are flexible but slow for math. For machine learning we use **NumPy arrays**, which are the Python version of a *vector* (1-D) or *matrix* (2-D). Arrays hold one numeric type and support **vectorized** math: operations apply to every element with no loop.

In [ ]:
v = np.array([1.0, 2.0, 3.0])
w = np.array([10.0, 20.0, 30.0])

print(v + w)        # elementwise
print(v * 2)        # broadcast a scalar
print(v @ w)        # dot product
print(v.mean(), v.std())

In [ ]:
# Compare: lists do NOT behave like vectors
print([1, 2, 3] + [10, 20, 30])    # concatenation, not addition!

In [ ]:
M = np.array([[1, 2, 3],
              [4, 5, 6]])
print(M)
print("shape:", M.shape, " dtype:", M.dtype)
print("row 1:", M[1], " element [1,2]:", M[1, 2], " column 0:", M[:, 0])
print(M.T)          # transpose

## 1.3 Dictionaries

A **dictionary** maps *keys* to *values*. Handy for named settings (hyperparameters!).

In [ ]:
config = {"learning_rate": 0.01, "epochs": 100, "model": "linear"}
print(config["epochs"])
config["epochs"] = 200
config

## 1.4 DataFrames

A **DataFrame** (from `pandas`) is a table: named columns, each a vector of one type. It is the workhorse object for tabular data.

In [ ]:
df = pd.DataFrame({
    "student": ["Ada", "Ben", "Cy", "Dee"],
    "hours_studied": [5.0, 2.5, 8.0, 4.0],
    "passed": [True, False, True, True],
})
df

In [ ]:
print(df.shape)            # (rows, columns)
print(df.dtypes)
df.head(2)

In [ ]:
print(df["hours_studied"])             # one column -> a Series (a labeled vector)
print(df["hours_studied"].mean())
print(df[df["hours_studied"] > 3])     # filter rows with a boolean condition
df["hours_squared"] = df["hours_studied"] ** 2   # add a column
df

### Quick check
Without running it: what is `type(df["passed"].iloc[0])`-ish? *(Hint: NumPy has its own bool type.)* Then run it. Python and NumPy/pandas each have their own versions of ints, floats, and bools; they usually interoperate, but you will occasionally see both.

In [ ]:
print(type(df["passed"].iloc[0]), type(True))

---
# 2. Pointers (references)

A variable does **not** contain an object; it **points to** one. Assignment copies the *pointer*, not the object.

Python doesn't expose raw pointers like C, but the idea is the same. `id()` shows an object's identity (its memory address in CPython), and `is` asks "same object?" (vs `==`, "equal values?").

In [ ]:
a = [1, 2, 3]
b = a            # b now points to the SAME list

print(id(a) == id(b), a is b)
b.append(99)
print("a:", a)   # a changed too!

```
a ──┐
    ├──▶ [1, 2, 3, 99]
b ──┘
```
Lists and arrays are **mutable**: they can be changed in place, so everyone pointing to them sees the change. To get an independent object, make an explicit **copy**.

In [ ]:
a = [1, 2, 3]
b = a.copy()     # new object with the same contents
b.append(99)
print("a:", a)
print("b:", b)
print(a is b, a == [1, 2, 3])

**Immutable** objects (ints, floats, strings, tuples) can't change in place, so sharing them is safe; 'modifying' creates a new object.

In [ ]:
s = "hello"
t = s
t = t + "!"      # builds a NEW string and repoints t
print(s, t)

The same trap exists with NumPy arrays and DataFrames, which matters a lot when you preprocess data.

In [ ]:
X = np.array([1.0, 2.0, 3.0])
Y = X                # same object
Y[0] = -100
print("X:", X)       # surprise: X was modified

X = np.array([1.0, 2.0, 3.0])
Z = X.copy()
Z[0] = -100
print("X:", X, " Z:", Z)

In [ ]:
# Slices of NumPy arrays are *views* (they share memory); slices of lists are copies
arr = np.arange(5)
view = arr[1:4]
view[0] = 999
print(arr)

lst = list(range(5))
sl = lst[1:4]
sl[0] = 999
print(lst)

In [ ]:
# Same story with DataFrames
d1 = pd.DataFrame({"a": [1, 2, 3]})
d2 = d1.copy()
d2.loc[0, "a"] = 100
print(d1["a"].tolist(), d2["a"].tolist())

### Pointers and functions
When you pass an object to a function, the function receives a pointer to it. A function that mutates a mutable argument changes the caller's object too. We'll see this again below.

> **Rule of thumb:** if you plan to modify data and still need the original, call `.copy()`.

---
# 3. Functions and methods

A **function** packages reusable code: it takes **arguments**, does something, and (usually) **returns** a value. Don't repeat yourself: write a function.

In [ ]:
def square(x):
    """Return x squared."""
    return x * x

print(square(4))
print(square(2.5))

Anatomy:

```python
def function_name(parameter1, parameter2=default):   # header ends with a colon
    # body is INDENTED (4 spaces)
    result = ...
    return result                                     # value handed back to the caller
```

In [ ]:
def mean_of(values):
    total = 0.0
    for v in values:
        total += v
    return total / len(values)

mean_of([88, 92, 79, 95])

**Default and keyword arguments:**

In [ ]:
def normalize(x, center=0.0, scale=1.0):
    return (x - center) / scale

print(normalize(10))
print(normalize(10, center=5, scale=2))     # keyword arguments: order doesn't matter
print(normalize(scale=2, x=10))

**Returning multiple values** (really, returning a tuple):

In [ ]:
def summarize(x):
    x = np.asarray(x)
    return x.mean(), x.std()

m, s = summarize([1, 2, 3, 4])
print(m, s)

**Scope:** variables created inside a function are *local*; they vanish when it returns.

In [ ]:
def f():
    local_var = 123
    return local_var

f()
# print(local_var)   # <- uncomment: NameError

**Functions and pointers**: mutation inside a function affects the caller's object; reassignment does not.

In [ ]:
def add_item_inplace(lst):
    lst.append("new")        # mutates the caller's list

def add_item_rebind(lst):
    lst = lst + ["new"]      # builds a new list; the caller's is untouched
    return lst

mine = ["a"]
add_item_inplace(mine); print(mine)
mine = ["b"]
add_item_rebind(mine);  print(mine)


## 3.1 Methods

A **method** is a function that *belongs to an object*, called with dot notation: `object.method(args)`. The object is passed implicitly as the first argument.

In [ ]:
s = "machine learning"
print(s.upper())
print(s.split(" "))
print(s.replace("machine", "deep"))

nums = [3, 1, 2]
nums.sort()
print(nums)

In [ ]:
arr = np.array([[1., 2.], [3., 4.]])
print(arr.sum(), arr.sum(axis=0), arr.mean())

df = pd.DataFrame({"g": ["a", "a", "b"], "y": [1.0, 3.0, 5.0]})
print(df.groupby("g")["y"].mean())

Equivalent function-style and method-style calls exist for many things (`len(x)` vs `x.shape`, `np.mean(a)` vs `a.mean()`). Use `?` or `help()` to read documentation, and `dir(obj)` to list an object's methods.

In [ ]:
help(str.split)

### Example: put it together
A function that standardizes a column (z-score), a very common ML preprocessing step.

In [ ]:
def standardize(values):
    """Return (values - mean) / std without modifying the input."""
    values = np.asarray(values, dtype=float)
    return (values - values.mean()) / values.std()

z = standardize(df["y"])
print(z, z.mean(), z.std())

---
# 4. Paths and loading data

Your code needs to find files on disk. A **path** is the address of a file or folder.

* **Absolute path:** starts from the root: `/Users/ada/projects/course/data/grades.csv` (Windows: `C:\Users\ada\...`)
* **Relative path:** starts from the **working directory**, i.e. where your code is currently "standing": `data/grades.csv`
* `.` = current folder, `..` = parent folder, `~` = your home folder

Absolute paths break when someone else runs your code on their machine. **Prefer relative paths inside a project folder.**

In [ ]:
cwd = Path.cwd()
print("Working directory:", cwd)
print("Parent:", cwd.parent)
print("Home:", Path.home())

Use `pathlib.Path` rather than gluing strings together: the `/` operator joins pieces and handles Mac/Windows differences for you.

In [ ]:
data_dir = Path("data")
file_path = data_dir / "grades.csv"
print(file_path)
print(file_path.name, file_path.suffix, file_path.parent)
print("exists?", file_path.exists())

To keep this notebook self-contained, we'll create a small CSV file ourselves, then load it back, just like you would with a real dataset.

In [ ]:
data_dir.mkdir(exist_ok=True)           # create folder if missing

df_out = pd.DataFrame({
    "student": ["Ada", "Ben", "Cy", "Dee", "Eli"],
    "hours_studied": [5.0, 2.5, 8.0, 4.0, np.nan],   # a missing value
    "exam_score": [88, 62, 95, 79, 70],
})
df_out.to_csv(file_path, index=False)
print("exists?", file_path.exists())
print(file_path.read_text())            # CSV is just text

In [ ]:
grades = pd.read_csv(file_path)
grades

In [ ]:
grades.info()

In [ ]:
print(grades.describe())
print(grades.isna().sum())     # count missing values per column

### Common errors
* `FileNotFoundError`: wrong path or wrong working directory. Debug with `Path.cwd()`, `path.exists()`, and `os.listdir()`.
* Windows backslashes in strings: `"C:\new\table.csv"` contains `\n` and `\t` escapes! Use `Path`, forward slashes, or raw strings `r"C:\new\table.csv"`.
* Spaces and weird characters in filenames cause headaches; prefer `snake_case_names.csv`.
* Other readers: `pd.read_excel`, `pd.read_json`, `pd.read_parquet`, `np.loadtxt`, `np.load`.

In [ ]:
try:
    pd.read_csv(data_dir / "does_not_exist.csv")
except FileNotFoundError as e:
    print("Caught:", e)

---
# 5. Floating point arithmetic

Computers store numbers in binary with finite memory. Integers are exact (Python's can be arbitrarily large), but **real numbers are approximated** using the IEEE 754 `float64` format: 1 sign bit, 11 exponent bits, 52 fraction bits (~15–16 significant decimal digits).

In [ ]:
0.1 + 0.2

In [ ]:
print(0.1 + 0.2 == 0.3)
print(f"{0.1:.20f}")
print(f"{0.2:.20f}")
print(f"{0.1 + 0.2:.20f}")
print(f"{0.3:.20f}")

**Why?** Just as 1/3 = 0.3333… has no finite decimal expansion, 0.1 has no finite *binary* expansion. The stored value is the nearest representable number, slightly off. Small errors accumulate.

In [ ]:
total = 0.0
for _ in range(10):
    total += 0.1
print(total, total == 1.0)

# Error accumulation
total = 0.0
for _ in range(1_000_000):
    total += 0.1
print(total)          # not exactly 100000

### How to compare floats
Never test floats for exact equality. Test whether they're *close*.

In [ ]:
a, b = 0.1 + 0.2, 0.3
print(abs(a - b) < 1e-9)
print(np.isclose(a, b))
import math
print(math.isclose(a, b))
print(np.allclose([a, 1.0], [b, 1.0]))   # for whole arrays

### Limits of the format

In [ ]:
print(np.finfo(float).eps)      # machine epsilon: gap between 1.0 and the next float
print(1.0 + 1e-16 == 1.0)       # too small to register!
print(1.0 + 2e-16 == 1.0)

print(np.finfo(float).max)      # largest float
print(np.finfo(float).tiny)     # smallest normal positive float
print(1e308 * 10)               # overflow -> inf
print(1e-320 / 1e10)            # underflow -> 0.0

In [ ]:
# Special values
print(np.inf - np.inf)          # nan: "not a number"
print(np.nan == np.nan)         # False! use np.isnan
print(np.isnan(np.nan))
print(-0.0 == 0.0)

### Gotchas that matter for machine learning

**1. Big + small = lost small.** Floats near 1e16 can't resolve differences of 1.

In [ ]:
big = 1e16
print((big + 1) - big)          # 0.0 not 1.0
print(2**53, float(2**53) + 1)

**2. Catastrophic cancellation.** Subtracting nearly equal numbers wipes out significant digits.

In [ ]:
x = 1e-8
print((1 - np.cos(x)) / x**2)           # true answer is 0.5
print(2 * np.sin(x / 2)**2 / x**2)      # algebraically identical, numerically stable

**3. Order of operations matters.** Floating point addition isn't associative.

In [ ]:
print((0.1 + 0.2) + 0.3, 0.1 + (0.2 + 0.3))
print(1e16 + 1.0 - 1e16, 1e16 - 1e16 + 1.0)

**4. Float32 vs float64.** Deep learning libraries often use 32-bit (or even 16-bit) floats for speed and memory, trading precision.

In [ ]:
print(np.float32(0.1) + np.float32(0.2) == np.float32(0.3))   # True here only because float32 rounds coarsely; errors are just bigger
print(np.finfo(np.float32).eps, np.finfo(np.float64).eps)
print(np.float32(16777216) + np.float32(1))   # 2**24: float32 can't count past here by 1s

**5. Overflow in `exp` and the log-sum-exp trick.** Softmax and likelihoods in ML are classic places to hit this.

In [ ]:
logits = np.array([1000.0, 1001.0, 1002.0])

with np.errstate(over="ignore", invalid="ignore"):
    naive = np.exp(logits) / np.exp(logits).sum()
print("naive :", naive)               # inf/inf -> nan

shifted = logits - logits.max()       # subtract the max: same answer mathematically
stable = np.exp(shifted) / np.exp(shifted).sum()
print("stable:", stable)

**Practical rules**

* Use `isclose`/`allclose`, never `==`, on computed floats.
* Be suspicious of `nan` and `inf`; they usually signal overflow, divide-by-zero, or log of zero or a negative number.
* Integers are exact; use them for counts and indices.
* Expect results to differ in the last few digits across machines and library versions. Set a random seed, and don't over-interpret differences at the 1e-12 level.